# 02 Asteria Simulation - Ballistic Descent

This notebook simulates the ARIS Asteria sounding rocket using RocketPy with **no parachutes** (ballistic descent).

In [ ]:
from matplotlib import pyplot as plt

from pipeline.dropout import DropoutInjector, DropoutMode, RandomDropout, WindowDropout
from pipeline.noise import NoiseInjector
from pipeline.noise.core.noise_profile import NoiseProfile
from pipeline.synthetic import SyntheticMagnetometer, SyntheticSensorGenerator
from rocketpy_simulation import RocketSimulation
from rocketpy_simulation.asteria import AsteriaMotor, AsteriaRocket
from rocketpy_simulation.environments import EuroCEnvironment

In [ ]:
SIMULATION_NAME: str = "02_asteria_ballistic"
DATA_PATH: str = "../../simulated_data/rocketpy/" + SIMULATION_NAME
plt.style.use('default')

## 1. Object Generation & Run
Instantiate the motor, rocket and environment builders, then run the full RocketPy flight simulation.
Parachutes are disabled (`include_parachutes=False`) so the rocket descends ballistically.

In [ ]:
motor = AsteriaMotor()
rocket = AsteriaRocket(motor=motor.build(), include_parachutes=False)
environment = EuroCEnvironment()

simulation = RocketSimulation(
    environment_builder=environment,
    rocket_builder=rocket,
)
result = simulation.run(name=SIMULATION_NAME)

## 2. Validation
Sanity-check the simulation result with rocket geometry, 3D trajectory and flight summary. Also previews the ground-truth dataframe.

In [ ]:
simulation.flight.rocket.draw()
simulation.flight.plots.trajectory_3d()
rocket = AsteriaRocket(motor=AsteriaMotor().build())
rocket.build().center_of_mass.plot()
simulation.flight.info()

In [ ]:
ground_truth = result.ground_truth.to_dataframe()
ground_truth.head(n=10)

## 3. Synthetic Sensor Addition
Add sensors that are not modeled by RocketPy (magnetometer) by computing their readings from the ground-truth trajectory.

In [ ]:
magnetometer_profile = NoiseProfile.from_yaml(
    "../../pipeline/noise/profiles/LSM303AGR_eCompass_magnetometer.yaml"
)
magnetometer = SyntheticMagnetometer(
    name="magnetometer",
    launch_site_elevation_m=environment.elevation,
)

generator = SyntheticSensorGenerator(sensors=[
    (magnetometer, NoiseInjector(magnetometer_profile), None),
])
generator.apply(result)

## 4. Noise Injection
Apply a hardware-derived noise profile to a clean sensor stream to produce a realistic noisy version.

In [ ]:
accel_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/LSM6DSO32_accelerometer.yaml")
gyro_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/LSM6DSO32_gyroscope.yaml")
barometer_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/MS5607_barometer.yaml")

accel_noisy = NoiseInjector(accel_noise_profile).apply(result.sensors_clean["accelerometer_clean"])
gyro_noisy = NoiseInjector(gyro_noise_profile).apply(result.sensors_clean["gyroscope_clean"])
barometer_noisy = NoiseInjector(barometer_noise_profile).apply(result.sensors_clean["barometer_clean"])

result.add_sensor_noisy("accelerometer", accel_noisy)
result.add_sensor_noisy("gyroscope", gyro_noisy)
result.add_sensor_noisy("barometer", barometer_noisy)

## 5. Dropout Simulation
Simulate sensor outages by masking samples with NaN.

In [ ]:
import numpy as np
import scipy.constants
from pipeline.dropout import DropoutInjector, DropoutMode, ExternalSignalThresholdDropout

dropout_start = result.metadata["out_of_rail_time"]
dropout_end = result.metadata["apogee_time"]

accel_mag = np.linalg.norm(result.ground_truth.acceleration, axis=1)
GNSS_MAX_ACCEL_G = 4.0
gnss_accel_dropout = DropoutInjector(
    strategies=[ExternalSignalThresholdDropout(
        reference_time=result.ground_truth.time,
        reference_values=accel_mag,
        threshold=GNSS_MAX_ACCEL_G * scipy.constants.g,
    )],
    mode=DropoutMode.NAN,
)

result.sensors_noisy["gnss"] = gnss_accel_dropout.apply(result.sensors_noisy["gnss"])

## 6. Export
Write the simulation result to CSV files and a pickle for downstream use.

In [ ]:
result.export_csv_data(DATA_PATH)
result.save(DATA_PATH + "/" + SIMULATION_NAME + ".pkl")